# 02 — Xem ảnh từ FPGA và đổi bộ lọc

Chạy `01_dma_verify.ipynb` PASS trước. Mở notebook này trên board và chọn **Kernel → Restart & Run All**. Notebook nạp overlay video; dừng ứng dụng MNIST trước khi chạy.

Mỗi ảnh output bên dưới được gửi qua **AXI DMA → FPGA → AXI DMA**. Ảnh “CPU tham chiếu” chỉ để đối chiếu. Preview Jupyter có truyền/mã hóa PNG, không đại diện cho tốc độ HDMI hay tốc độ core.


In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'video_driver.py').is_file():
    raise RuntimeError('Mở notebook trong thư mục video_demo đã giải nén trên board, có video_driver.py cạnh notebook.')
sys.path.insert(0, str(ROOT))
RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)
print('Thư mục demo:', ROOT)


In [ ]:
from video_driver import VideoAccelerator
from board_workflows import assert_pixels
verification_path = RESULTS / 'board_dma_results.json'
if not verification_path.is_file():
    raise RuntimeError('Chạy 01_dma_verify.ipynb trước.')
verification = json.loads(verification_path.read_text(encoding='utf-8'))
if verification.get('status') != 'PASS' or not verification.get('physical_board_verified'):
    raise RuntimeError('Kiểm chứng board chưa PASS; xem results/board_dma_results.json.')
accelerator = VideoAccelerator.load(ROOT / 'overlays')
if verification['overlay_manifest']['bit_sha256'] != accelerator.fingerprint['bit_sha256']:
    accelerator.close()
    raise RuntimeError('Bitstream đã đổi: chạy lại notebook 01.')
print('Backend: FPGA AXI DMA | FCLK0:', accelerator.clock_mhz, 'MHz')


In [ ]:
import base64, io, time
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import display, HTML, clear_output
from reference_filters import apply_filter, rgb_for_display, MODE_NAMES

def make_frame(index=0, width=640, height=480):
    x=np.arange(width)[None,:]
    y=np.arange(height)[:,None]
    rgb=np.empty((height,width,3),np.uint8)
    rgb[:,:,0]=(x+index*5)%256
    rgb[:,:,1]=(y+index*3)%256
    rgb[:,:,2]=((x//40+y//40)%2)*255
    image=Image.fromarray(rgb)
    draw=ImageDraw.Draw(image)
    bx=(index*13)%max(1,width-100)
    draw.rectangle((bx,80,bx+90,170),fill=(255,255,255))
    draw.ellipse((width//2-60,height//2-60,width//2+60,height//2+60),fill=(0,0,0))
    draw.line((0,height-40,width-1,30),fill=(255,255,255),width=4)
    return np.asarray(image)[:,:,::-1].copy()

def image_data(image):
    stream=io.BytesIO()
    Image.fromarray(rgb_for_display(image)).save(stream,format='PNG')
    return 'data:image/png;base64,'+base64.b64encode(stream.getvalue()).decode('ascii')

def show_frame(mode=3, threshold=128, index=0, image=None):
    source=make_frame(index) if image is None else image
    actual=accelerator.process(source,int(mode),int(threshold))
    golden=apply_filter(source,int(mode),int(threshold))
    assert_pixels(actual,golden,'preview')
    figures=[]
    for label,array in [('Ảnh đầu vào',source),('Output FPGA — '+MODE_NAMES[int(mode)],actual),('CPU tham chiếu — đối chiếu',golden)]:
        figures.append('<figure style="margin:8px;flex:1"><figcaption>'+label+'</figcaption><img style="max-width:100%" src="'+image_data(array)+'"></figure>')
    display(HTML('<div style="display:flex;flex-wrap:wrap">'+''.join(figures)+'</div>'))
    metrics=accelerator.last_metrics.copy()
    print('Pixel: PASS | core %.3f ms | DMA+control %.3f ms | cả driver %.3f ms' % (
        metrics['core_active_ms'],metrics['dma_and_control_ms'],metrics['total_driver_ms']))
    return actual

_ = show_frame(3)


## Điều khiển bộ lọc
Nếu image có `ipywidgets`, menu dưới đây cập nhật output FPGA. Nếu menu không hiện, chạy cell thủ công `show_frame(1)` hoặc `show_frame(2, threshold=128)`; không cần cài thêm thư viện để dùng cách thủ công.


In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    print('Không có ipywidgets. Dùng show_frame(mode, threshold, index) trực tiếp.')
else:
    mode_control=widgets.Dropdown(options=[(MODE_NAMES[m],m) for m in range(4)],value=3,description='Bộ lọc')
    threshold_control=widgets.IntSlider(value=128,min=0,max=255,step=1,description='Ngưỡng',continuous_update=False)
    frame_control=widgets.IntSlider(value=0,min=0,max=100,step=1,description='Frame',continuous_update=False)
    widget_output=widgets.Output()
    def update_widget(change=None):
        with widget_output:
            clear_output(wait=True)
            show_frame(mode_control.value,threshold_control.value,frame_control.value)
    for control in (mode_control,threshold_control,frame_control): control.observe(update_widget,names='value')
    display(widgets.VBox([mode_control,threshold_control,frame_control,widget_output]))
    update_widget()


## Thử ảnh của em
Upload PNG/JPEG vào thư mục `video_demo`, rồi sửa tên trong cell dưới. Ảnh được resize về 640×480 trên ARM trước khi gửi FPGA; resize không phải tác vụ của IP bộ lọc.


In [ ]:
IMAGE_PATH = None  # Ví dụ: 'anh_cua_em.jpg'
if IMAGE_PATH:
    input_rgb=Image.open(ROOT / IMAGE_PATH).convert('RGB').resize((640,480),Image.Resampling.BILINEAR if hasattr(Image,'Resampling') else Image.BILINEAR)
    input_bgr=np.asarray(input_rgb)[:,:,::-1].copy()
    _=show_frame(3,image=input_bgr)
else:
    print('Ảnh cá nhân là tùy chọn; các ảnh kiểm thử và demo có sẵn trong gói.')


## Preview chuyển động tùy chọn
Cell dưới mặc định không chạy vòng lặp. Đổi `RUN_ANIMATION=True` để xem 50 frame tổng hợp, mỗi frame thực sự qua FPGA. Đo nhịp toàn vòng preview riêng; nó gồm tạo ảnh, đối chiếu, PNG và truyền notebook. Đây không phải nguồn camera hoặc HDMI trực tiếp.


In [ ]:
RUN_ANIMATION = False
if RUN_ANIMATION:
    started=time.perf_counter()
    for frame_index in range(50):
        clear_output(wait=True)
        _=show_frame(3,index=frame_index)
        time.sleep(0.1)
    print('Nhịp vòng preview %.2f frame/s; gồm hiển thị, khác tốc độ FPGA.' % (50/(time.perf_counter()-started)))


## Khi kết thúc
Chạy `accelerator.close()` trong một cell để dừng DMA và giải phóng bộ đệm. Trước khi chuyển về MNIST hoặc nạp overlay khác, dừng preview và đóng accelerator; chỉ một overlay chạy trên cùng board tại một thời điểm.
